# report04 — 조명원 — **WiFi / LTE / 5G 파형**

> ### ❓ 이 리포트가 답하는 질문
> **어떤 통신 신호로 드론을 비추며, 각 신호의 장단점은?**

### ⚡ 결론부터 (TL;DR)

1. **패시브 레이더는 남의 송신기를 빌려 쓴다.** 자기 신호가 없으니, 아무 셀이나 **늘 켜 두는 '상시 신호' 딱 하나**로만 표적을 본다 — LTE = **CRS**, 5G = **SSB**, WiFi = **프리앰블(VHT-LTF)**. 어떤 신호를 빌리느냐가 이 레이더의 눈을 통째로 정한다.
2. **신호가 넓을수록 거리를 잘 가른다.** 두 물체를 거리로 나누는 능력(거리분해능 ΔR)은 기준신호가 **실제로 켜는 주파수 폭**($B_{ref}$)이 정한다 — 유휴 셀에서 실측: WiFi **76.6 MHz → ΔR 3.9 m**, LTE **18.0 MHz → ΔR 16.7 m**, 5G SSB **7.2 MHz → ΔR 41.6 m**.
3. **신호가 자주 나올수록 빠른 표적을 본다.** 헷갈리지 않고 볼 수 있는 최고 속도(무모호 속도 $v_{max}$)는 기준신호가 **초당 몇 번 반복되나**(PRF)가 정한다 — WiFi/LTE 는 1 kHz 로 나와 각각 $v_{max}$ 14·41 m/s, 5G SSB 는 50 Hz 로만 나와 **1.07 m/s** 뿐.
4. **5G 는 패시브 레이더에 불리하다 — 좁고, 드물다.** SSB 는 좁아서(ΔR 42 m) 42 m 안쪽 물체를 한 덩어리로 뭉뚱그리고, 드물어서(20 ms 마다 한 번) 걷는 속도(약 1.4 m/s)만 넘어도 놓친다. 채널은 98 MHz 나 되는데 패시브가 쓸 수 있는 건 그중 **7.3%** 뿐이다.
5. **뜻밖에 구세대 LTE 가 더 좋은 조명등이다.** CRS 는 넓고(전대역) 자주(1 kHz) 나와 거리·속도 두 축 모두 5G 를 앞선다 — ΔR 16.7 m vs 41.6 m, $v_{max}$ 41 vs 1.1 m/s. **세대가 최신이라고 패시브 레이더에 좋은 건 아니다.**

### 🗺️ 어디부터 읽나

| 절 | 무엇을 |  |
|---|---|---|
| §1 | 패시브는 왜 '상시 신호'만 쓰나 — 표준별 기준신호 | 빌려 쓰는 자의 제약. LTE=CRS · 5G=SSB · WiFi=프리앰블 |
| §2 | 두 성질이 눈을 정한다 — **넓이=거리, 반복=속도** | ΔR 은 $B_{ref}$, $v_{max}$ 는 PRF. 유휴 셀 G1 실측표 |
| §3 | 자원격자로 눈으로 확인 + **5G 가 왜 불리한가** | 좁다 × 드물다. LTE 와의 대비 |
| 바쁘면 | §2 의 G1 표와 §3 | 이 리포트의 결론 |

---


## 🔰 5분이면 이해하는 이 리포트

*(수식·표가 부담스러우면 이 칸만 읽어도 됩니다. 아래 §들은 같은 이야기를 숫자로 증명합니다.)*

**패시브 레이더는 손전등이 없는 탐정입니다.** 보통 레이더는 자기가 전파를 쏘고 그 반사를 봅니다. 패시브 레이더는 그걸 못 합니다 — 자기 손전등이 없거든요. 대신 **누군가 이미 켜 둔 불빛**, 그러니까 주변 기지국이나 와이파이가 늘 내보내는 신호가 드론에 튕겨 돌아오는 걸 봅니다.

**남의 불빛을 빌려 쓰려면 조건이 하나 있습니다 — 그 불빛이 '어떻게 생겼는지' 미리 알아야** 합니다. 튕겨 온 빛을 알아보려면 원래 모습과 비교해야 하니까요. 그런데 통신 신호 대부분은 실어 나르는 **데이터가 매번 바뀌어** 미리 알 수가 없습니다. 그래서 패시브 레이더는 **언제 들어도 똑같아서 미리 아는 '기준 불빛'** 만 쓸 수 있습니다. 이 상시 기준 불빛은 통신 표준마다 딱 하나씩입니다 — 옛 LTE 엔 **CRS**, 최신 5G 엔 **SSB**, 와이파이엔 패킷 앞머리(프리앰블)가 그것입니다.

**그리고 이 기준 불빛이 어떻게 생겼느냐가 탐정의 눈을 통째로 정합니다.** 두 가지가 중요합니다.
- **불빛이 넓은 주파수를 쓸수록** 가까이 붙은 두 물체를 촘촘히 갈라 봅니다 (거리 구분).
- **불빛이 자주 깜빡일수록** 빠르게 움직이는 물체를 놓치지 않습니다 (속도 구분).

**여기서 최신 5G 가 뜻밖에 나쁩니다.** 5G 가 늘 켜 두는 기준 불빛(SSB)은 **좁고**(그래서 42 m 안쪽 물체는 한 덩어리로 뭉쳐 보임), **20 ms 에 한 번만** 깜빡입니다(그래서 걸음 속도보다 빠른 드론은 놓침). 반대로 구세대 LTE 는 넓고 자주 켜져서 거리도 속도도 훨씬 잘 봅니다. 비유하자면, **남이 켜 주는 손전등으로 드론을 찾는데 — 5G 손전등은 빛줄기가 좁고 그나마 가끔씩만 켜 준다**는 겁니다. 통신은 5G 가 빠를지 몰라도, 패시브 레이더의 조명등으로는 옛 LTE 가 오히려 낫습니다.

> **한 줄 요약** — 패시브 레이더는 남의 신호를 빌려 봅니다. 그 신호가 **넓고 자주 켜질수록** 잘 봅니다. 최신 5G 의 상시 신호는 **좁고 드물어서** 오히려 옛 LTE 보다 못합니다.

## 📋 이 결과가 어디서 어떻게 나왔나

> 이 절은 **직접 참여하지 않은 사람도 출처를 따라가고 재현할 수 있도록** 넣었습니다. 버전·GPU 는 노트북 생성 시점에 **실제로 읽어온 값**입니다.

### 1️⃣ 무엇을 참고했나

| 항목 | 출처 | 성격 |
|---|---|---|
| LTE CRS / 5G SSB / WiFi VHT-LTF 자원격자 배치 | 3GPP TS 36.211 · 3GPP TS 38.211 · IEEE 802.11ac → `src/waveforms.py`. 조사 근거는 `docs/waveform_research.json` | 🔴 우리 구현 (Sionna 에 WiFi/LTE/SSB **자원격자가 없다**) |
| 5G NR 뉴머롤로지 (μ·SCS·슬롯·CP) | **`sionna.phy.nr.CarrierConfig`** — Sionna 에게 물어본 값 (표를 손으로 안 짬) | 🟢 라이브러리 (3GPP TS 38.211 구현) |
| ΔR · $v_{max}$ 폐형식 | $\Delta R_b$ = c/$B_{ref}$ (바이스태틱; 모노 등가 c/2B) · $v_{max}$ = PRF·λ/4 — 교과서 레이더 공식 | 📐 해석식 |

### 2️⃣ 어떤 도구가 무엇을 했나 — **Sionna 내부인가, 우리가 짠 건가**

| 도구 | 하는 일 | 어디서 도는가 |
|---|---|---|
| `sionna-phy` | Sionna PHY (`ofdm`/`nr`/`channel`) — OFDM 변복조 · 3GPP 뉴머롤로지 · RT 경로를 신호에 적용 | 🟢 **Sionna 내부** (PyTorch 백엔드, GPU) |
| `matplotlib` | matplotlib — 도표·그래프 | 🔴 **별도** (CPU). 계산 결과를 *그리기만* 한다 |

> 🔑 **이 구분이 이 프로젝트에서 가장 자주 오해받는 지점입니다.**
> - **전파**(경로·지연·도플러·렌더·라디오맵)는 🟢 **Sionna 가** 합니다.
> - **표적 RCS** 는 🟡 우리가 짠 **SBR** 이 합니다 — Sionna 에 RCS 솔버가 없기 때문입니다. 다만 광선추적은 Sionna 가 쓰는 **Mitsuba 3 엔진을 그대로** 씁니다.
> - **레이더 신호처리**(ECA/CFAR)는 🔴 우리가 짰습니다 — Sionna 에 레이더 DSP 가 없습니다.

### 3️⃣ 라이브러리 (실행 시점 **실측** 버전)

| 라이브러리 | 버전 | 무엇에 쓰나 |
|---|---|---|
| `sionna` | 2.0.1 | 광선추적(RT) + PHY(OFDM/NR/채널) — **이 프로젝트의 중심** |
| `numpy` | 2.5.0 | 수치 계산 전반 |
| `scipy` | 1.18.0 | 스플라인(단면 보간·암 경로) · STFT(스펙트로그램) |
| `matplotlib` | 3.11.0 | 도표 |

### 4️⃣ 어디서 돌렸나

- **Python** 3.12.13 · Linux 5.15.0-136-generic
- **GPU** — `src/gpu.py` 가 **여유 메모리를 보고 자동 선택**합니다 (하드코딩 없음):
  - 0, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
  - 1, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
  - 2, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
  - 3, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
- `CUDA_VISIBLE_DEVICES` = (고정 안 함 — src/gpu.py 가 여유 메모리 보고 자동 선택)

- **계산 비용**: 이 리포트는 무거운 재측정을 하지 않는다 — 파형 격자를 만들어 제원을 읽는 것은 CPU 초 단위. 그림은 report2 파이프라인이 이미 남긴 것을 재사용한다. 노트북 생성도 초 단위.

### 5️⃣ 어떻게 다시 돌리나 (재현)

```bash
cd /home/yunjung/workspace/sionna2

# 파형 제원을 만들어서 재고 그림·JSON 을 남긴다 (report2 파이프라인 재사용, 재측정 없음)
~/.venvs/py312/bin/python src/viz_report2.py        # 측정 + 그림 + JSON
~/.venvs/py312/bin/python src/make_notebook04.py    # JSON -> report04.ipynb

# 파형 제원을 직접 눈으로 확인하고 싶으면:
~/.venvs/py312/bin/python src/waveforms.py          # 표준별 B_ref · PRF · ΔR · v_max
```

### 6️⃣ 본문 숫자는 어디서 오나

이 노트북의 **숫자는 손으로 적지 않았습니다.** 측정 스크립트가 JSON 을 남기고, 노트북 생성기(`src/make_notebook*.py`)가 그 JSON 을 읽어 본문에 주입합니다. → **그림과 글이 어긋날 수 없습니다.** 숫자가 이상하면 JSON 을 보세요.

### 7️⃣ 무엇이 산출되나

| 산출물 | 무엇 |
|---|---|
| `outputs/report2_waveform_rcs.json` | **이 리포트의 모든 파형 숫자.** 측정 원본 |
| `outputs/figures/report2_ref_signal.png` | §2 넓이→거리 · 반복→속도 예산 그림 |
| `outputs/figures/report2_resource_grid.png` | §3 자원격자 사진 (WiFi/LTE/5G × 유휴/풀로드) |
| `outputs/figures/report2_occupancy.png` | §3 점유율은 늘어도 거리분해능은 안 늘더라 |

### 8️⃣ ⚠️ 믿으면 안 되는 것 (신뢰 경계)

> 정직함이 이 프로젝트의 규칙입니다. **아래는 이 리포트가 보장하지 않는 것들입니다.**

- **§2 의 ΔR·$v_{max}$ 는 '유휴 셀'을 전제한다.** 유휴 셀은 상시 기준신호만 내보내므로 패시브가 기댈 수 있는 **가장 정직한 기본선**이다. 부하가 걸린 셀에서 수신기가 송신 파형 전체를 직접 받아 기준으로 쓸 수 있다면(captured full-waveform) 대역은 채널 전대역까지 넓어질 수 있으나, 그건 **다른 전제**다 — 두 경우를 섞어 인용하지 말 것.
- **PRS 로 넓힌 5G 수치는 낙관적 상한이다.** PRS 는 상시 신호가 아니라 **측위 세션이 설정돼야** 켜지는 옵션이며, 남의 셀을 빌려 쓰는 패시브 수신기는 그것이 켜져 있다고 가정할 수 없다. 그래서 §2 의 기본선은 SSB(7.2 MHz)이지 PRS(전대역)가 아니다.
- **WiFi 의 PRF 1 kHz 는 '혼잡한 AP' 대표값이다.** 트래픽이 없으면 비콘(약 100 ms 주기)만 남아 반복이 훨씬 느려진다 — WiFi 의 속도 성능은 트래픽에 크게 의존한다.
- **이 리포트는 파형이 무엇을 주는가(대역·반복·분해능)만 다룬다.** 그 파형이 정말 규격대로 만들어졌는지의 증명은 → report05, 표적이 얼마나 밝은지(RCS)는 → report06 소관이다.

### 9️⃣ 앞뒤 리포트

| 리포트 | 관계 |
|---|---|
| [report03](report03.ipynb) — 표적 검증 | **앞 리포트.** '무엇을 볼 것인가'(드론)를 확정했다. 여기서는 '무엇으로 볼 것인가'(조명원) |
| **report04 (여기)** — 조명원 · 파형 | 패시브가 빌려 쓰는 상시 신호 3종과 그 장단점 — 넓이(거리)·반복(속도) |
| [report05](report05.ipynb) — 파형 검증 | **다음 리포트.** 여기서 만든 파형이 정말 3GPP/IEEE 규격대로인지 Sionna 로 대조한다 |

<details><summary><b>🔤 용어집 — 모르는 말이 나오면 여기</b> (클릭)</summary>

| 용어 | 뜻 |
|---|---|
| **패시브 레이더** | 자기 송신기 없이 **남의 신호(방송·통신)의 반사**로 표적을 보는 레이더 |
| **조명원(illuminator)** | 패시브 레이더가 빌려 쓰는 '남의 송신기' — 기지국·와이파이 AP 등 |
| **기준신호(reference signal)** | 패시브가 상관을 걸 때 쓰는 '미리 아는 신호'. 표준마다 상시 하나 |
| **상시(always-on) 신호** | 임의의 셀이 **언제나** 내보낸다고 믿을 수 있는 신호. LTE=CRS, 5G=SSB, WiFi=프리앰블 |
| **CRS** | LTE Cell-specific Reference Signal — 매 서브프레임(1 ms)·채널 전대역에 상시 나오는 셀 기준신호 |
| **SSB** | 5G NR SS/PBCH Block — 유휴 gNB 가 늘 내보내는 유일한 신호. 좁고(중앙 20 RB) 드물다(20 ms 주기) |
| **VHT-LTF** | WiFi 802.11ac 패킷 앞머리(프리앰블)의 롱 트레이닝 필드. 어떤 패킷에도 붙지만 반복은 트래픽에 의존 |
| **점유대역 $B_{ref}$** | 기준신호가 자원격자에서 **실제로 켜는** 주파수 폭. 채널 전대역과 다르다 |
| **거리분해능 ΔR** | 두 표적을 거리로 가르는 최소 간격. **바이스태틱** $\Delta R_b$=c/$B_{ref}$ (우리 시스템), 모노스태틱 등가는 c/2B(절반). 넓은 신호일수록 촘촘 |
| **PRF** | pulse repetition frequency — 기준신호가 **초당 몇 번** 반복되나. 자주일수록 빠른 표적을 봄 |
| **무모호 속도 $v_{max}$** | 헷갈리지 않고 볼 수 있는 최대 속도. $v_{max}$ = PRF·λ/4 |
| **자원격자(resource grid)** | 가로=시간·세로=주파수의 모눈종이. OFDM 신호는 이 칸을 채워 만든다 |
| **RE (resource element)** | 자원격자의 **한 칸** (한 부반송파 × 한 OFDM 심볼) |
| **OFDM** | 부반송파 수백 개에 데이터를 잘게 나눠 싣는 변조 방식. 통신 신호의 기본 골격 |
| **PRS** | Positioning Reference Signal — **측위 세션이 설정돼야** 켜지는 옵션. 상시 신호가 아니다 |
| **도플러 접힘(aliasing)** | 반복이 너무 드물어 빠른 표적의 속도를 헷갈리는 현상. $v_{max}$ 를 넘으면 생긴다 |

</details>

---


> **앞 리포트**(report03)에서 우리는 **무엇을 볼 것인가** — 표적 드론의 3D 모델을 실물·커뮤니티 CAD 와 대조해 확정했습니다. 이 리포트는 그 반대쪽, **무엇으로 볼 것인가** — 패시브 레이더가 빌려 쓰는 **조명원(illuminator, 남의 송신기)의 파형**을 다룹니다.

---
## §1. 패시브는 왜 '상시 신호'만 쓸 수 있나

> 🔍 **여기서 하는 일** — 패시브 레이더가 남의 신호 중 **어떤 것**을 빌릴 수 있는지, 그리고 왜 표준마다 선택지가 딱 하나뿐인지 짚습니다.

패시브 레이더(자기 송신기 없이 남의 신호의 반사로 표적을 보는 레이더)는 송신기를 **빌려 씁니다.** 빌려 쓰는 데엔 두 가지 제약이 따라붙습니다.

1. **미리 아는 신호여야 한다.** 튕겨 온 신호가 표적인지 알아보려면 '원래 신호'와 비교(상관)해야 합니다. 그런데 통신 신호가 나르는 **데이터는 매 순간 바뀌므로** 수신기가 미리 알 수 없습니다. 미리 아는 부분 — 즉 내용이 고정된 **기준신호** 만 상관에 쓸 수 있습니다.
2. **아무 셀이나 늘 내보내야 한다.** 특정 조건에서만 켜지는 신호는, 하필 그 순간 그 셀이 안 켜면 표적을 놓칩니다. 그래서 임의의 셀이 **언제나** 내보낸다고 믿을 수 있는 **상시(always-on) 신호** 여야 합니다.

이 두 조건을 동시에 만족하는 신호는 표준마다 **딱 하나씩**입니다:

| 표준 | 상시 기준신호 | 왜 이것뿐인가 |
|---|---|---|
| **LTE** | **CRS** (Cell-specific Reference Signal) | 매 서브프레임(1 ms)·채널 전대역에 늘 뿌린다 |
| **5G NR** | **SSB** (SS/PBCH Block) | NR 에는 **CRS 같은 상시 전대역 신호가 없다.** 유휴 기지국이 늘 내보내는 건 SSB 뿐 |
| **WiFi** | **프리앰블 (VHT-LTF)** | 모든 패킷 앞머리에 붙는다 — 단 **반복 횟수가 트래픽에 의존** |

> ⚠️ **PRS(측위 기준신호)는 상시 신호가 아닙니다.** 넓은 대역을 쓰지만 **측위 세션이 설정돼야** 켜지는 옵션이라, 남의 셀을 빌려 쓰는 패시브 수신기는 그것이 켜져 있다고 가정할 수 없습니다. 그래서 이 리포트의 기본선은 어디까지나 위 3개의 **상시** 신호입니다.

> 💡 **한 줄로** — 패시브가 빌릴 수 있는 조명은 '언제 봐도 똑같고(미리 앎), 늘 켜져 있는(상시)' 신호뿐이고, 그건 표준마다 하나뿐입니다. 다음 절에서 이 하나의 선택이 레이더의 눈을 어떻게 정하는지 봅니다.

---
## §2. 두 가지 성질이 눈을 정한다 — 넓이(거리)와 반복(속도)

> 🔍 **여기서 하는 일** — 빌린 신호의 **주파수 넓이**와 **반복 빈도**가 각각 거리·속도를 얼마나 잘 보는지 정한다는 것을, 유휴 셀에서 실측한 표로 보입니다.

### 2.1 넓을수록 거리를 잘 가른다

> **비유** — 손전등 빛줄기가 굵으면 두 물체가 한 덩어리로 뭉쳐 보이고, 가늘고 날카로울수록 둘을 따로 짚습니다. 레이더에서 이 '날카로움'을 정하는 게 신호의 **주파수 넓이**입니다.

두 표적을 거리로 가르는 최소 간격을 **거리분해능 ΔR** 이라 합니다(우리는 **바이스태틱**이라 $\Delta R_b$=c/$B_{ref}$; 모노스태틱 등가는 절반). 여기서 결정적인 것은 채널이 차지한 전체 대역이 **아니라**, 기준신호가 자원격자에서 **실제로 켜는 주파수 폭** — **점유대역 $B_{ref}$** 입니다.

$$\Delta R_b = \frac{c}{B_{ref}}\quad(\text{바이스태틱; 모노 등가 } c/2B)$$

$B_{ref}$ 가 넓을수록 ΔR 이 작아져(=촘촘해져) 가까이 붙은 두 물체를 갈라 봅니다.

### 2.2 자주 나올수록 빠른 표적을 본다

> **비유** — 빠르게 지나가는 물체를 카메라로 잡으려면 셔터를 자주 눌러야 합니다. 드문드문 찍으면 그 사이 물체가 어디로 얼마나 갔는지 헷갈립니다.

표적이 움직이면 되돌아오는 신호의 주파수가 살짝 밀립니다(도플러). 이 밀림으로 속도를 재는데, 기준신호가 **초당 몇 번 반복되나**(PRF)가 헷갈리지 않고 볼 수 있는 최고 속도 — **무모호 속도 $v_{max}$** 를 정합니다.

$$v_{max} = \frac{\mathrm{PRF}\cdot\lambda}{4}$$

반복이 드물면($v_{max}$ 가 작으면) 그보다 빠른 표적은 **도플러가 접혀(aliasing)** 엉뚱한 속도로 읽힙니다.

### 2.3 유휴 셀에서 실측 — 세 조명등을 나란히

`src/waveforms.py` 가 3GPP·IEEE 스펙대로 격자를 만들고, 그 격자에서 $B_{ref}$·PRF 를 그대로 읽어 ΔR·$v_{max}$ 를 계산합니다 (아래 숫자는 손으로 적은 게 아니라 **격자에서 측정**한 값):

| 표준 | 기준신호 | 채널 대역 | **$B_{ref}$** | **ΔR** | **PRF** | **$v_{max}$** |
|---|---|---|---|---|---|---|
| WiFi 802.11ac | VHT-LTF | 76 MHz | **76.6 MHz** | **3.9 m** | 1000 Hz | 14.4 m/s |
| LTE Rel-9 | CRS | 18 MHz | **18.0 MHz** | **16.7 m** | 1000 Hz | 41 m/s |
| **5G NR Rel-16** | **SSB** | 98 MHz | **7.2 MHz** | **41.6 m** | **50 Hz** | **1.07 m/s** |

![reference signal budget](outputs/figures/report2_ref_signal.png)

*(그림 (a): 회색 = 채널 전체 대역, 색 = 패시브가 실제로 쓰는 기준신호 대역. 5G 만 둘이 크게 다릅니다. (b): 그 대역이 정하는 거리분해능. (c): 거리(가로)·속도(세로)를 한 점으로 — 오른쪽·아래일수록 나쁨.)*

**세 줄로 읽는 법:**
- **WiFi** 는 기준신호가 이미 넓어(**76.6 MHz**) 거리를 가장 촘촘히 봅니다(**ΔR 3.9 m**). 다만 5 GHz 대라 도달 거리가 짧고, 반복이 트래픽에 의존합니다.
- **LTE** 는 CRS 가 채널 전대역(**18.0 MHz**)에 매 1 ms 뿌려져 거리(**ΔR 16.7 m**)·속도(**$v_{max}$ 41 m/s**) 모두 균형이 좋습니다.
- **5G SSB** 는 채널이 98 MHz 나 되는데도 상시 켜는 건 중앙 20 RB **7.2 MHz** 뿐 — 거리가 거칠고(**ΔR 41.6 m**), 20 ms 마다 한 번만 나와 속도도 약합니다(**$v_{max}$ 1.07 m/s**). 다음 절에서 자세히 봅니다.

In [ ]:
# §2 재현 — waveforms.py 가 격자에서 읽은 속성을 그대로 출력 (본문 숫자에 하드코딩 없음)
import sys; sys.path.insert(0, 'src')
from waveforms import always_on_waveforms

print(f"{'표준':16s} {'기준신호':9s} {'B_ref':>10} {'ΔR':>8} {'PRF':>9} {'v_max':>9}")
for k, wf in always_on_waveforms().items():          # 유휴 셀 = 상시 기준신호만
    print(f'{wf.name:16s} {wf.ref_name:9s} {wf.ref_bw_hz/1e6:7.2f}MHz '
          f'{wf.range_resolution_m:6.2f}m {wf.pilot_rate_hz:7.0f}Hz {wf.v_unambiguous_ms:7.2f}m/s')

---
## §3. 눈으로 확인 — 자원격자, 그리고 5G 가 불리한 이유

> 🔍 **여기서 하는 일** — 세 신호가 격자에서 **실제로 어떤 칸을 켜는지** 사진으로 보고, 왜 5G 가 패시브 레이더에 불리한지 두 축(거리·속도)으로 못박습니다.

### 3.1 자원격자로 본 세 신호

통신 신호는 **자원격자(가로=시간, 세로=주파수인 모눈종이)** 의 칸(RE)을 채워 만듭니다. 아래 그림에서 **위 줄은 유휴 셀**(상시 기준신호만), **아래 줄은 풀로드 셀**(데이터까지 꽉 채운 상태)입니다.

![resource grid](outputs/figures/report2_resource_grid.png)

위 줄만 보면 됩니다 — 패시브가 쓸 수 있는 건 그것뿐이니까요. **WiFi 프리앰블**은 세로(주파수)로 꽉 차 있고(넓다), **LTE CRS** 도 전대역에 흩뿌려져 있는데, **5G SSB** 만 세로로 **가운데 작은 블록**에 몰려 있습니다 — 이 좁음이 곧 거친 거리분해능입니다.

> 💡 아래 줄(풀로드)에서 회색으로 꽉 찬 칸(PDSCH/DATA)은 **데이터**입니다. 에너지는 많지만 수신기가 **모르는 내용**이라 상관에 못 씁니다 — 그래서 셀이 바빠져도 패시브가 쓸 '아는 신호'의 넓이는 거의 안 늘어납니다.

이 파형이 정말 3GPP·IEEE 규격대로 만들어졌는지의 증명은 여기서 하지 않습니다 — **→ report05** 에서 Sionna 로 교차대조합니다. 여기서는 그 파형이 **무엇을 켜고, 그래서 무엇을 주는가**만 봅니다.

### 3.2 셀이 바빠져도 거리분해능은 안 늘더라

'셀이 데이터로 꽉 차면 신호가 넓어져 거리를 더 잘 보지 않을까?' — 만들어서 재보면 그렇지 않습니다.

![occupancy](outputs/figures/report2_occupancy.png)

격자 점유율(왼쪽)은 유휴→풀로드로 5G 기준 1.4% → 80% 로 수십 배 뛰지만, **거리분해능(오른쪽)은 WiFi·LTE 는 거의 안 움직입니다** — 상시 기준신호가 **이미 넓기** 때문입니다. 늘어난 건 상관에 못 쓰는 데이터 에너지뿐입니다.

> ⚠️ 5G 만 오른쪽에서 41.6 m → 3.1 m 로 급전환하는데, 그건 **PRS(측위 기준신호)가 켜졌기 때문**입니다. PRS 는 상시 신호가 아니라 **측위 세션을 가정한 것**이므로, 이 급전환은 **낙관적 상한**이지 패시브가 늘 기댈 수 있는 기본선이 아닙니다. 기본선은 어디까지나 SSB(7.2 MHz)입니다.

### 3.3 5G 가 불리한 이유 — 좁고, 드물다

**① 좁다 (거리가 거칠다).** SSB 는 채널 한가운데 20 RB = **7.2 MHz** 만 켭니다. 그래서 ΔR = **41.6 m** — 42 m 안쪽에 있는 두 물체는 **한 덩어리**로만 보입니다. 채널 자체는 **98 MHz** 나 되는데 패시브가 상시로 쓸 수 있는 건 그중 **7.3%** 뿐입니다.

**② 드물다 (빠른 표적을 놓친다).** SSB 는 SS 버스트가 **20 ms 주기**로만 나옵니다 → PRF **50 Hz** → $v_{max}$ **1.07 m/s**. 걷는 속도(약 1.4 m/s)만 넘어도 도플러가 접혀 속도를 헷갈립니다.

이 **두 불리함이 겹친 것**이 5G 가 패시브 레이더에 약한 이유입니다.

### 3.4 대비 — 구세대 LTE 가 오히려 좋은 조명등

LTE CRS 는 **전대역**(18.0 MHz)을 켜고 **매 서브프레임(1 kHz)** 나옵니다. 그래서 거리·속도 두 축 모두 5G 를 앞섭니다:

| 축 | 무엇이 정하나 | LTE CRS | 5G SSB | LTE 가 유리한 정도 |
|---|---|---|---|---|
| **거리** ΔR | 점유대역 $B_{ref}$ | **16.7 m** (18.0 MHz) | 41.6 m (7.2 MHz) | 약 2.5× 촘촘 |
| **속도** $v_{max}$ | PRF | **41 m/s** (1000 Hz) | 1.07 m/s (50 Hz) | 약 38× 빠름 |

> **정리** — 통신 세대가 최신이라고 패시브 레이더에 좋은 조명등이 되는 건 아닙니다. 오히려 5G 는 늘 켜 두는 기준신호가 **좁고 드물어서** 옛 LTE 보다 못합니다. **세대와 조명 품질은 별개**입니다. 쓸 조명원을 고를 땐, 통신 성능이 아니라 **상시 신호의 넓이와 반복**을 봐야 합니다.

![.](outputs/renders/anim/spectrum_nr.gif)

<sub>5G NR 스펙트럼 — 상시 기준신호(SSB) 점유 대역이 좁아 거리분해능이 거칠다.</sub>

![.](outputs/renders/anim/spectrum_lte.gif)

<sub>LTE 스펙트럼 — CRS 가 전대역(18.0 MHz)을 켜 거리분해능이 촘촘하다(ΔR_b 16.7 m, 5G SSB 의 약 2.5배 촘촘).</sub>

---
> **다음 리포트**: report05 — 지금까지 '이 파형은 이런 대역·반복을 준다'고 했는데, 그 파형이 정말 3GPP·IEEE 규격대로 만들어졌는지 **Sionna PHY 로 교차대조**해 확인합니다.